In [5]:
import torch
from lightning import LightningModule, LightningDataModule
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from pydantic import BaseModel, model_validator
from typing_extensions import Self, List, Dict, Tuple
from jaxtyping import Float, Int, Bool
import numpy as np


Tensor = torch.Tensor
Array = np.ndarray

In [6]:
class InformerDataset(Dataset):
    def __init__(self, covariates: Float[Tensor, 'datapoints covariates'], timestamps: Float[Tensor, 'datapoints time_feats'],
                 targets: Float[Tensor, 'datapoints targets'], lookback_len: int, label_len: int, pred_len: int) -> None:
        """
        Initializes the dataset.

        Args:
            covariates: Tensor of covariate features [num_datapoints, num_covariates].
                        These are features known potentially in advance.
            timestamps: Tensor of timestamp features [num_datapoints, num_time_features].
                        Derived features from the time index (e.g., month, day).
            targets: Tensor of target features [num_datapoints, num_targets].
                     The value(s) the model aims to predict.
            lookback_len: Length of the input sequence for the encoder (L).
                          How far back the model looks.
            label_len: Length of the known sequence prefix for the decoder input (Lb).
                       Part of the lookback window used as context for the decoder.
            pred_len: Length of the sequence to predict (P).
                      The prediction horizon.
        """
        
        super().__init__()
        self.covariates = covariates
        self.timestamps = timestamps
        self.targets = targets
        self.lookback_len = lookback_len
        self.label_len = label_len
        self.pred_len = pred_len

        # get num of targets
        self.num_targets = targets.shape[-1]

        # Total features for encoder/decoder data inputs (covariates + targets)
        self.num_features = covariates.shape[-1] + targets.shape[-1]

        # Calculate number of slidable windows
        self.num_samples = len(covariates) - lookback_len - pred_len + 1

    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx: int) -> Tuple[Float[Tensor, 'lookback_len covariates+targets'], Float[Tensor, 'lookback_len time_feats'],
                                             Float[Tensor, 'label_len+pred_len covariates+targets'], Float[Tensor, 'label_len+pred_len time_feats'],
                                             Float[Tensor, 'pred_len targets']]:
        """
        Generates one sample of data for the Informer model.

        Args:
            idx: The index of the sample window.

        Returns:
            A tuple containing:
            - encoder_features: Features for the encoder input [L, Feat].
            - encoder_timestamps: Timestamps for the encoder input [L, TimeFeat].
            - decoder_features_full: Features for the decoder input [Lb+P, Feat].
            - decoder_timestamps_full: Timestamps for the decoder input [Lb+P, TimeFeat].
            - targets: Ground truth target values for the prediction horizon [P, Targ].
              (L=lookback_len, Lb=label_len, P=pred_len)
        """    

        # 1. Calculate the seq idxs
        enc_start_idx: int = idx
        enc_end_idx: int = idx + self.lookback_len

        dec_known_start_idx: int = enc_end_idx - self.label_len
        dec_known_end_idx: int = enc_end_idx

        dec_pred_start_idx: int = enc_end_idx
        dec_pred_end_idx: int = enc_end_idx + self.pred_len

        target_seq_start_idx: int = enc_end_idx
        target_seq_end_idx: int = enc_end_idx + self.pred_len

        # 2. Get encoder inputs
        encoder_covariates: Float[Tensor, 'lookback_len covariates'] = self.covariates[enc_start_idx : enc_end_idx]
        encoder_timestamps: Float[Tensor, 'lookback_len time_feats'] = self.timestamps[enc_start_idx : enc_end_idx]
        encoder_targets: Float[Tensor, 'lookback_len targets'] = self.targets[enc_start_idx : enc_end_idx]

        # Combine covariates and targets for encoder data input
        encoder_features: Float[Tensor, 'lookback_len covariates+targets'] = torch.cat([encoder_covariates, encoder_targets], dim=-1)

        # 3. Decoder Inputs
        # Slice KNOWN covariates and targets (label_len) for decoder
        decoder_covariates_known: Float[Tensor, 'label_len covariates'] = self.covariates[dec_known_start_idx : dec_known_end_idx]
        decoder_targets_known: Float[Tensor, 'label_len targets'] = self.targets[dec_known_start_idx : dec_known_end_idx]

        # Combine known covariates and targets
        decoder_features_known: Float[Tensor, 'label_len covariates+targets'] = torch.cat([decoder_covariates_known, decoder_targets_known], dim=-1)

        # Prediction part (pred_len) - known future covariates, zeroed targets
        decoder_covariates_pred: Float[Tensor, 'pred_len covariates'] = self.covariates[dec_pred_start_idx : dec_pred_end_idx]

        # Create placeholder ONLY for the target columns
        decoder_targets_pred_placeholder: Float[Tensor, 'pred_len targets'] = torch.zeros((self.pred_len, self.num_targets),
                                                                        dtype=self.targets.dtype,
                                                                        device=self.targets.device)

        # combine decoder prediction targets and covates to prediction features
        decoder_features_pred: Float[Tensor, 'pred_len covariates+targets'] = torch.cat([decoder_covariates_pred, decoder_targets_pred_placeholder], dim=-1)

        decoder_features: Float[Tensor, 'label_len+pred_len covariates+targets'] = torch.cat([decoder_features_known, decoder_features_pred], dim=0)
        decoder_timestamps: Float[Tensor, 'label_len+pred_len time_feats'] = self.timestamps[dec_known_start_idx:dec_pred_end_idx]

        # 5. Ground Truth Target
        # Slice only targets for the prediction period
        targets: Float[Tensor, 'pred_len targets'] = self.targets[target_seq_start_idx : target_seq_end_idx]


        return encoder_features, encoder_timestamps, decoder_features, decoder_timestamps, targets

        

In [7]:
def sin_cos_encoding(x, x_max):
    sin_encoded = np.sin(2 * np.pi * x / x_max)
    cos_encoded = np.cos(2 * np.pi * x / x_max)
    return sin_encoded, cos_encoded

In [ ]:
class DataModule(LightningDataModule):
    def __init__(self, hparams_struct):
        super().__init__()
        self.save_hyperparameters(hparams_struct.model_dump())
        # print(self.hparams)

    def setup(self, stage=None):
        # read in data and get dfs for covariates, timestamps and targets
        df: pd.DataFrame = pd.read_csv(self.hparams.path)

        # extarte targets and features and convert them to numpy
        covariate_array: Float[Array, 'datapoints covariates'] = df[self.hparams.covariates].to_numpy()
        target_array: Float[Array, 'datapoints targets'] = df[self.hparams.target].to_numpy()[:, None] # unsqueze to adhere to pytorch target convetion

  
        # Extrat the time stamp data
        dates_series_str = df[self.hparams.date_column]

        # Get date column and save it as datetime obj
        dates_series_datetime: pd.Series = pd.to_datetime(dates_series_str, format='%Y-%m-%d %H:%M:%S', errors='raise')

        # extract covariate times from date df
        month_array: Int[Array, 'datapoints'] = dates_series_datetime.dt.month.to_numpy() - 1 # minus 1 for zero indexed months
        day_array: Int[Array, 'datapoints']  = dates_series_datetime.dt.dayofweek.to_numpy()
        hour_array: Int[Array, 'datapoints'] = dates_series_datetime.dt.hour.to_numpy()

        timestamps_array: Int[Array, 'datapoints timestamp'] = np.stack((month_array, day_array, hour_array), axis=1)


        # split into train val and test dataset not randomly becaue time matters
        # dataset sizes
        n_samples = len(covariate_array)
        n_train = int(self.hparams.train_split * n_samples)
        n_val = int(self.hparams.val_split * n_samples)

        # split dataframe
        train_covariates = covariate_array[:n_train]
        val_covariates = covariate_array[n_train : n_train + n_val]
        test_covariates = covariate_array[n_train + n_val :]

        train_timestamp = timestamps_array[:n_train]
        val_timestamp = timestamps_array[n_train : n_train + n_val]
        test_timestamp = timestamps_array[n_train + n_val :]

        train_target = target_array[:n_train]
        val_target = target_array[n_train : n_train + n_val]
        test_target = target_array[n_train + n_val :]

        # Calcualte mean and std of training set to use for scaling
        self.train_feat_mean = train_covariates.mean()
        self.train_feat_std = train_covariates.std() + 1e-6

        self.train_target_mean = train_target.mean()
        self.train_target_std = train_target.std() + 1e-6

        # scale non time featues
        train_covariates_scaled = (train_covariates - self.train_feat_mean) / self.train_feat_std
        val_covariates_scaled = (val_covariates - self.train_feat_mean) / self.train_feat_std
        test_covariates_scaled = (test_covariates - self.train_feat_mean) / self.train_feat_std

  
        # scale targets
        train_target_scaled = (train_target - self.train_target_mean) / self.train_target_std
        val_target_scaled = (val_target - self.train_target_mean) / self.train_target_std
        test_target_scaled = (test_target - self.train_target_mean) / self.train_target_std

        # turn into tensors
        train_covariates_tensor = torch.tensor(train_covariates_scaled, dtype=torch.float32, device=self.hparams.dataset_device)
        val_covariates_tensor = torch.tensor(val_covariates_scaled, dtype=torch.float32, device=self.hparams.dataset_device)
        test_covariates_tensor = torch.tensor(test_covariates_scaled, dtype=torch.float32, device=self.hparams.dataset_device)

        train_timestamp_tensor = torch.tensor(train_timestamp, dtype=torch.long, device=self.hparams.dataset_device)
        val_timestamp_tensor = torch.tensor(val_timestamp, dtype=torch.long, device=self.hparams.dataset_device)
        test_timestamp_tensor = torch.tensor(test_timestamp, dtype=torch.long, device=self.hparams.dataset_device)

        train_targets_tensor = torch.tensor(train_target_scaled, dtype=torch.float32, device=self.hparams.dataset_device)
        val_targets_tensor = torch.tensor(val_target_scaled, dtype=torch.float32, device=self.hparams.dataset_device)
        test_targets_tensor = torch.tensor(test_target_scaled, dtype=torch.float32, device=self.hparams.dataset_device)

        self.train_dataset = InformerDataset(train_covariates_tensor, train_timestamp_tensor, train_targets_tensor,
                                             self.hparams.lookback_len,
                                             self.hparams.label_len,
                                             self.hparams.pred_len)
        
        self.val_dataset = InformerDataset(val_covariates_tensor, val_timestamp_tensor, val_targets_tensor,
                                        self.hparams.lookback_len,
                                        self.hparams.label_len,
                                        self.hparams.pred_len)
        
        self.test_dataset = InformerDataset(test_covariates_tensor, test_timestamp_tensor, test_targets_tensor,
                                        self.hparams.lookback_len,
                                        self.hparams.label_len,
                                        self.hparams.pred_len)

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.hparams.batch_size, num_workers=self.hparams.num_workers, shuffle=True, pin_memory=self.hparams.pin_memory)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.hparams.batch_size, num_workers=self.hparams.num_workers, pin_memory=self.hparams.pin_memory)

    def test_dataloader(self):
        return DataLoader(self.test_dataset, batch_size=self.hparams.batch_size, num_workers=self.hparams.num_workers, pin_memory=self.hparams.pin_memory)
        
    def predict_dataloader(self):
        return DataLoader(self.test_dataset, batch_size=self.hparams.batch_size, num_workers=self.hparams.num_workers, pin_memory=self.hparams.pin_memory)
        